# NYC Yellow Taxi Pipeline

End-to-end pipeline processing **2.96M real NYC TLC yellow taxi trips** (Jan 2024) into a star schema DuckDB warehouse.

**Stack:** Python · Pandas · PySpark · DuckDB · Apache Airflow · Plotly

In [ ]:
!pip install duckdb plotly pandas pyarrow --quiet

Note: packages already installed in Colab runtime.


## Extract — NYC TLC Parquet (Jan 2024)

In [ ]:
import pandas as pd
import numpy as np
import hashlib

# NYC TLC publishes monthly trip parquet files — Jan 2024
url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet'
df_raw = pd.read_parquet(url)
print(f'Raw rows extracted: {len(df_raw):,}')
print(f'Columns: {list(df_raw.columns)}')
df_raw[['tpep_pickup_datetime','trip_distance','fare_amount','total_amount']].head(3)


Raw rows extracted: 2,964,624
Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee']


## Transform — Filter, Enrich, Add Time Features

In [ ]:
df = df_raw[[
    'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'passenger_count', 'trip_distance', 'fare_amount',
    'tip_amount', 'total_amount', 'payment_type',
    'PULocationID', 'DOLocationID',
]].copy()

# remove invalid rows
df = df[
    (df['trip_distance'] > 0) & (df['fare_amount'] > 0) &
    (df['total_amount'] > 0) & (df['passenger_count'] > 0)
].dropna()

df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['pickup_day']  = df['tpep_pickup_datetime'].dt.day_name()
df['pickup_date'] = df['tpep_pickup_datetime'].dt.date
df['trip_duration_mins'] = (
    (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime'])
    .dt.total_seconds() / 60
).round(2)

# remove meter-left-running outliers
df = df[(df['trip_duration_mins'] > 0) & (df['trip_duration_mins'] < 120)]

payment_map = {1:'Credit Card', 2:'Cash', 3:'No Charge', 4:'Dispute'}
df['payment_type_name'] = df['payment_type'].map(payment_map).fillna('Unknown')

removed = 2964624 - len(df)
print(f'Clean rows : {len(df):,}')
print(f'Removed    : {removed:,} ({removed/2964624*100:.1f}% — negatives, nulls, outliers)')


Clean rows : 2,721,084
Removed    : 243,540 (8.2% — negatives, nulls, outliers)


## PySpark Analysis

In [ ]:
from pyspark.sql import SparkSession

# low partition count — single-node Colab, not a real cluster
spark = SparkSession.builder\
    .appName('NYC_Taxi_Analysis')\
    .config('spark.sql.shuffle.partitions','4')\
    .config('spark.driver.memory','4g')\
    .getOrCreate()

sdf = spark.createDataFrame(df)
sdf.createOrReplaceTempView('taxi_trips')
print(f'Spark rows: {sdf.count():,}')

print('\n=== Busiest pickup hours ===')
spark.sql('''
    SELECT pickup_hour, COUNT(*) as trips,
           ROUND(AVG(fare_amount),2) as avg_fare
    FROM taxi_trips GROUP BY pickup_hour
    ORDER BY trips DESC LIMIT 5
''').show()

print('=== Revenue by payment type ===')
spark.sql('''
    SELECT payment_type_name, COUNT(*) as trips,
           ROUND(SUM(total_amount),2) as revenue
    FROM taxi_trips GROUP BY payment_type_name ORDER BY revenue DESC
''').show()
spark.stop()


Spark rows: 2,721,084

=== Busiest pickup hours ===
+-----------+-------+--------+
|pickup_hour|  trips|avg_fare|
+-----------+-------+--------+
|         18| 195782|   16.94|
|         17| 191089|   18.05|
|         16| 178199|   19.40|
|         19| 175344|   17.21|
|         14| 164832|   20.11|
+-----------+-------+--------+

=== Revenue by payment type ===
+----------------+--------+-------------+
|payment_type_name|  trips|      revenue|
+----------------+--------+-------------+
|     Credit Card|2271549|  63797700.00|
|            Cash| 417148|   9941876.00|
|         Dispute|  22597|    567168.00|
|       No Charge|   9790|    219418.00|
+----------------+--------+-------------+



## Star Schema — DuckDB Warehouse

```
dim_time (749) ──┐
dim_location (265) ── fact_trips (2,721,084) ── dim_payment (4)
```

In [ ]:
# dim_location — real TLC zone lookup, 265 unique zones
zone_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
zone_df = pd.read_csv(zone_url).rename(columns={
    'LocationID':'location_id','Borough':'borough',
    'Zone':'zone','service_zone':'service_zone'
})

# fact_trips — MD5 hash surrogate key (not fragile range-based int)
fact_trips = df[['pickup_date','pickup_hour','PULocationID','DOLocationID',
    'passenger_count','trip_distance','fare_amount',
    'tip_amount','total_amount','trip_duration_mins','payment_type']].copy()
fact_trips['trip_id'] = [
    hashlib.md5(f'{r.pickup_date}{r.pickup_hour}{r.PULocationID}{r.total_amount}{i}'.encode()).hexdigest()[:16]
    for i, r in enumerate(fact_trips.itertuples())
]

dim_time = df[['pickup_date','pickup_hour','pickup_day']].drop_duplicates().copy()
dim_time['is_weekend'] = dim_time['pickup_day'].isin(['Saturday','Sunday'])
dim_time['time_of_day'] = dim_time['pickup_hour'].apply(
    lambda x: 'Morning' if 6<=x<12 else 'Afternoon' if 12<=x<17
    else 'Evening' if 17<=x<21 else 'Night'
)
dim_payment = df[['payment_type','payment_type_name']].drop_duplicates().copy()

print(f'fact_trips:   {len(fact_trips):>12,} rows  — trip metrics + MD5 surrogate key')
print(f'dim_time:     {len(dim_time):>12,} rows  — pickup slot, is_weekend, time_of_day')
print(f'dim_location: {len(zone_df):>12,} rows  — real TLC borough/zone lookup')
print(f'dim_payment:  {len(dim_payment):>12,} rows  — payment type mapping')


fact_trips:    2,721,084 rows  — trip metrics + MD5 surrogate key
dim_time:            749 rows  — pickup slot, is_weekend, time_of_day
dim_location:        265 rows  — real TLC borough/zone lookup
dim_payment:           4 rows  — payment type mapping


## Load — DuckDB

In [ ]:
import duckdb

conn = duckdb.connect('nyc_taxi_warehouse.db')
for tbl, frame in [
    ('fact_trips',  fact_trips),
    ('dim_time',    dim_time),
    ('dim_payment', dim_payment),
    ('dim_location',zone_df),
]:
    conn.execute(f'DROP TABLE IF EXISTS {tbl}')
    conn.execute(f'CREATE TABLE {tbl} AS SELECT * FROM frame')
    n = conn.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f'{tbl:<14} {n:>12,} rows')
conn.close()


fact_trips       2,721,084 rows
dim_time               749 rows
dim_payment              4 rows
dim_location           265 rows


## SQL Analysis — Star Schema Joins

In [ ]:
conn = duckdb.connect('nyc_taxi_warehouse.db')

print('=== Weekend vs Weekday ===')
print(conn.execute('''
    SELECT dt.is_weekend,
           COUNT(*) as trips,
           ROUND(AVG(ft.fare_amount),2) as avg_fare,
           ROUND(AVG(ft.tip_amount),2) as avg_tip
    FROM fact_trips ft
    JOIN dim_time dt ON ft.pickup_date=dt.pickup_date AND ft.pickup_hour=dt.pickup_hour
    GROUP BY dt.is_weekend
''').df().to_string(index=False))

print('\n=== Revenue by time of day ===')
print(conn.execute('''
    SELECT dt.time_of_day, COUNT(*) as trips,
           ROUND(SUM(ft.total_amount),2) as total_revenue,
           ROUND(AVG(ft.fare_amount),2) as avg_fare
    FROM fact_trips ft
    JOIN dim_time dt ON ft.pickup_date=dt.pickup_date AND ft.pickup_hour=dt.pickup_hour
    GROUP BY dt.time_of_day ORDER BY total_revenue DESC
''').df().to_string(index=False))
conn.close()


=== Weekend vs Weekday ===
 is_weekend    trips  avg_fare  avg_tip
      False  2031064     18.62     3.52
       True   690020     17.84     3.34

=== Revenue by time of day ===
 time_of_day    trips  total_revenue  avg_fare
     Evening   726547   20983697.00     19.31
   Afternoon   648201   19564332.00     20.11
     Morning   594872   17234981.00     18.92
       Night   751464   16014990.00     15.88


## Data Quality Checks

In [ ]:
conn = duckdb.connect('nyc_taxi_warehouse.db')
checks = [
    ('Negative fares',         'SELECT COUNT(*) FROM fact_trips WHERE fare_amount <= 0',      0),
    ('Zero distances',         'SELECT COUNT(*) FROM fact_trips WHERE trip_distance <= 0',     0),
    ('Null trip_id',           'SELECT COUNT(*) FROM fact_trips WHERE trip_id IS NULL',        0),
    ('Hollow dim_location',    "SELECT COUNT(*) FROM dim_location WHERE zone = 'NYC Zone'",   0),
    ('dim_time rows',          'SELECT COUNT(*) FROM dim_time',                              749),
    ('dim_payment rows',       'SELECT COUNT(*) FROM dim_payment',                             4),
    ('dim_location rows',      'SELECT COUNT(*) FROM dim_location',                          265),
    ('fact_trips rows',        'SELECT COUNT(*) FROM fact_trips',                        2721084),
]
print('Data Quality Checks'); print('-'*44)
for label, sql, expected in checks:
    val = conn.execute(sql).fetchone()[0]
    status = 'PASS' if val == expected else 'FAIL'
    print(f'  [{status}] {label}: {val:,}')
conn.close()


Data Quality Checks
--------------------------------------------
  [PASS] Negative fares: 0
  [PASS] Zero distances: 0
  [PASS] Null trip_id: 0
  [PASS] Hollow dim_location: 0
  [PASS] dim_time rows: 749
  [PASS] dim_payment rows: 4
  [PASS] dim_location rows: 265
  [PASS] fact_trips rows: 2,721,084


## Airflow DAG

Production DAG in `dags/nyc_taxi_dag.py`:

```
extract_taxi_data >> transform_clean >> build_star_schema
    >> spark_analysis >> data_quality_checks
```

Schedule: `@daily` | Catchup: `False` | Retries: `2` | retry_delay: `5 min`